# 01 - Data Preprocessing

Prepare the raw customer churn dataset for modeling: inspect quality, encode categorical columns, split the data, scale numerical features, and save reusable artifacts.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

In [ ]:
import joblib

## 2. Load Raw Data

Read the original dataset and standardize the target column name.

In [ ]:
df = pd.read_csv(r"D:\project\data\raw\customer_churn.csv")

print(df.shape)

In [ ]:
df.rename(columns={'ch': 'churn'}, inplace=True)

## 3. Initial Data Inspection

Review sample rows, schema, descriptive statistics, missing values, and categorical columns.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum() / len(df) * 100).round(1)

In [ ]:
num_cols = [
    'avg_monthly_gb',
    'credit_score',
    'num_complaints',
    'annual_income',
    'customer_satisfaction'
]

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df[num_cols].isnull().sum()

In [ ]:
df.select_dtypes(include='object').columns

## 4. Encode Categorical Features

Use ordinal mappings for ordered categories and one-hot encoding for nominal categories.

In [ ]:
df = df.drop(columns=['customer_id', 'signup_date'])

In [ ]:
df['contract']= df['contract'].map({'month_to_month': 0, 'one_year': 1, 'two_year': 2})

In [ ]:
df['paperless_billing'].value_counts()

In [ ]:
df['paperless_billing'] = df['paperless_billing'].map({'Yes': 1, 'No': 0})

In [ ]:
df['paperless_billing'].value_counts()

In [ ]:
df['education'].value_counts()

In [ ]:
df['education'] = df['education'].map({
    'high_school': 0,
    'college': 1,
    'bachelor': 2,
    'master': 3,
    'phd': 4
})

In [ ]:
df['education'].value_counts()

In [ ]:
df = pd.get_dummies(df, columns=[
    'gender',
    'marital_status',
    'payment_method'
])

## 5. Validate Encoded Dataset

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

## 6. Duplicates and Outliers

Check duplicate rows and summarize potential numerical outliers using the IQR method.

In [ ]:
df.duplicated().sum()

In [ ]:
# Numerical columns

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

print(f"Number of numerical columns: {len(numerical_cols)}")
numerical_cols.tolist()

In [ ]:
# Detect outliers using IQR

outlier_summary = []

for col in numerical_cols:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]
    
    outlier_summary.append({
        'Feature': col,
        'Outlier Count': len(outliers),
        'Outlier Percentage': round((len(outliers)/len(df))*100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df.sort_values(
    by='Outlier Percentage',
    ascending=False
).head(15)

In [ ]:
bool_cols = df.select_dtypes(
    include='bool'
).columns

df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
print(df.shape)
df.head()

## 7. Save Cleaned Dataset

In [ ]:
df.to_csv(r"D:\project\data\processed\clean_data_.csv", index=False)

## 8. Train-Test Split

Separate features from the target and create a stratified train/test split.

In [ ]:
X = df.drop(columns=['churn'])
y = df['churn']

In [ ]:
import sklearn
print("sklearn version:", sklearn.__version__)

In [ ]:
X_train, X_test, y_train,y_test = sklearn.model_selection.train_test_split(
    X,y,test_size= 0.2 , random_state=42 ,stratify=y
)

In [ ]:
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Testing samples : {X_test.shape[0]:,}")

print(f"Train churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate : {y_test.mean()*100:.2f}%")

## 9. Feature Scaling

Fit `StandardScaler` on the training data only, then transform train and test numerical features.

In [ ]:
df.info()

In [ ]:
scale_cols = [
    'age',
    'annual_income',
    'dependents',
    'tenure',
    'monthlycharges',
    'totalcharges',
    'num_services',
    'customer_satisfaction',
    'num_complaints',
    'num_service_calls',
    'late_payments',
    'avg_monthly_gb',
    'days_since_last_interaction',
    'credit_score'
]

In [ ]:
scaler = sklearn.preprocessing.StandardScaler()

In [ ]:
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
X_train.head()

## 10. Save Processed Data and Scaler

Combine the processed train/test data for downstream notebooks and save the fitted scaler.

In [ ]:
train_df = X_train.copy()
train_df['churn'] = y_train.values

In [ ]:
test_df = X_test.copy()
test_df['churn'] = y_test.values

In [ ]:
clean_df = pd.concat([train_df, test_df], ignore_index=True)

In [ ]:
clean_df.to_csv(r"D:\project\data\processed\clean_data.csv", index=False)

In [ ]:
joblib.dump(scaler, r"D:\project\models\scaler\scaler.pk1")